In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# PIPELINE FILE: p1_bronze_pipeline
# PURPOSE      : DLT Bronze layer — ingests raw data from two source types
#                into Unity Catalog bronze schema as streaming Delta tables.
#
# THIS IS NOT A REGULAR NOTEBOOK.
# Do not run it with "Run All". It must be attached to a Delta Live Tables
# (Lakeflow Spark Declarative Pipelines) pipeline configuration in the UI.
# Databricks DLT runtime executes this file — it orchestrates the streaming,
# checkpointing, schema inference, and table creation automatically.
#
# SOURCES:
#   1. landing.streaming_orders_raw  → Delta table (written by 02_order_producer)
#   2. /Volumes/.../historical_orders/ → CSV files (written by 01_generate_upload_data)
#   3. /Volumes/.../reviews/           → CSV files (written by 01_generate_upload_data)
#
# OUTPUTS (3 Bronze streaming tables):
#   bronze.orders                 ← live POS orders
#   bronze.historical_orders_raw  ← 6-month historical batch
#   bronze.reviews_raw            ← customer reviews
#
# WHAT "BRONZE" MEANS:
#   Bronze = raw data as-is from the source. No transformations, no joins,
#   no business logic. Data is stored exactly as it arrived.
#   Purpose: preserve the original record for audit, replay, and debugging.
#   If something goes wrong in Silver, you replay from Bronze — not the source.
# ═══════════════════════════════════════════════════════════════════════════════
import dlt
# Note: 'import dlt' is the classic API name.
# As of 2025, Databricks renamed DLT to "Lakeflow Spark Declarative Pipelines".
# The new import is: from pyspark import pipelines as dp
# Both work identically. We use 'import dlt' to match the project transcript.

from pyspark.sql.functions import *

CATALOG  = "restaurant_catalog"
VOL_PATH = f"/Volumes/{CATALOG}/landing/raw_files"

In [0]:
# ─── TABLE 1: LIVE STREAMING ORDERS ───────────────────────────────────────────
#
# WHAT YOU'RE DOING:
#   Reading the Delta table streaming_orders_raw as a continuous stream.
#   Every time 02_order_producer appends a new batch of orders (new Delta
#   commit/version), this DLT table will automatically pick up those new rows.
#
# WHY @dlt.table DECORATOR:
#   This is the "declarative" part of Spark Declarative Pipelines.
#   You declare WHAT the table should contain (the function's return value).
#   You do NOT write: create table, manage checkpoint, handle failures.
#   DLT handles all of that automatically in the background.
#
# WHY pipelines.reset.allowed = false:
#   CRITICAL production setting. Without it, if anyone clicks
#   "Full Refresh" on the DLT pipeline UI, it would WIPE this Bronze
#   table and re-process everything from scratch.
#   For a streaming table in production that receives millions of events,
#   a full refresh = hours of downtime and potential data loss.
#   Setting this to false prevents accidental full resets.
#   Always set this on streaming Bronze tables.
#
# WHY maxFilesPerTrigger = 1:
#   Controls how many Delta commits the stream processes per trigger.
#   Setting it to 1 means: process one producer run at a time.
#   This prevents the pipeline from "catching up" all at once and
#   consuming too many resources, giving you predictable, controlled ingestion.
# ──────────────────────────────────────────────────────────────────────────────

@dlt.table(
    name    = "orders",
    comment = "Bronze: live POS orders streaming from landing.streaming_orders_raw Delta table. "
              "Replaces Azure Event Hub + Kafka connector in production. "
              "DLT checkpoints ensure each order is processed exactly once.",
    table_properties = {
        "quality"                 : "bronze",
        "pipelines.reset.allowed" : "false",
    }
)
def orders():
    return (
        spark.readStream
             .format("delta")
             .option("maxFilesPerTrigger", 1)
             .table(f"{CATALOG}.landing.streaming_orders_raw")
    )

In [0]:
# ─── TABLE 2: HISTORICAL ORDERS via AUTO LOADER ────────────────────────────────
#
# WHAT YOU'RE DOING:
#   Using Auto Loader (cloudFiles format) to incrementally ingest
#   historical_orders CSV files from the Unity Catalog Volume.
#   This replaces LakeFlow Connect CDC from Azure SQL in the original project.
#
# HOW AUTO LOADER WORKS (cloudFiles):
#   Auto Loader tracks which files it has already processed using a
#   checkpoint directory. On each pipeline run it checks:
#   "Are there any new CSV files in this folder I haven't processed yet?"
#   If yes → read and ingest only those new files.
#   If no  → do nothing.
#   This is called "incremental file ingestion" — exactly what LakeFlow
#   Connect does for SQL tables, but for files.
#
# WHY cloudFiles.schemaLocation:
#   Auto Loader infers the CSV schema on first run and saves it to
#   this path. On subsequent runs, it reuses the saved schema instead
#   of re-inferring — this is faster and prevents schema drift issues
#   where a column type changes unexpectedly between runs.
#
# WHY cloudFiles.schemaEvolutionMode = addNewColumns:
#   If new columns appear in the CSV in the future (e.g., someone adds
#   a "discount_code" column), Auto Loader will automatically add it to
#   the Bronze table schema instead of failing.
#   Production rule: always set schema evolution mode explicitly.
#   Options: addNewColumns (safe) | failOnNewColumns (strict) | rescue (logs unknowns)
#
# WHY cloudFiles.inferColumnTypes = true:
#   Without this, Auto Loader reads all CSV columns as StringType.
#   With it: total_amount becomes DoubleType, timestamp becomes TimestampType, etc.
#   This saves you from doing cast() on every column in Silver.
# ──────────────────────────────────────────────────────────────────────────────

@dlt.table(
    name="historical_orders_raw",
    comment = "Bronze",
    table_properties = {
        "quality":"bronze"
    }
)
def historical_orders_raaw():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format","csv")
        .option("cloudFiles.schemaLocation",f"{VOL_PATH}/_checkpoints/historical_orders_schema")
        .option("cloudFiles.inferColumnTypes","true")
        .option("cloudFiles.schemaEvolutionMode","addNewColumns")
        .option("header","true")
        .load(f"{VOL_PATH}/historical_orders/")
    )

In [0]:
@dlt.table(
    name = "review_raw",
    comment = "Bronze: customer review data ingested via Auto Loader (cloudFiles) "
              "from Unity Catalog Volume. Will be enriched with AI sentiment "
              "analysis (ai_query) in Silver pipeline.",
    table_properties = {
        "quality":"bronze"
    }
)
def reviews_raw():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format","csv")
        .option("cloudFiles.schemaLocation",
                f"{VOL_PATH}/_checkpoints/reviews_schema")
        .option("cloudFiles.schemaEvolutionMode","addNewColumns")
        .option("cloudFiles.inferColumnTypes","true")
        .option("header","true")
        .load(f"{VOL_PATH}/reviews/")
    )